# Campanhas persistentes de modelos

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="05_treinamento_campanha"


{'mode': 'READ_EXISTING',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': '8a186e1f-3eb9-4819-b6ec-f99ff331853e'}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Catálogo de campanhas e orçamento. Identidades hi.demo são experimentos sintéticos separados sobre o snapshot, não 500 fontes acadêmicas reais.

In [2]:
display(pd.DataFrame(call(cfg,"GET","/v1/training/campaigns")))
from hashlib import sha256
prefix=sha256(params.request_key.encode()).hexdigest()[:10]
model_keys=[f"hi.demo.campaign.{prefix}.{i:03}" for i in range(params.model_count)]
display({"model_count":len(model_keys),"max_parallel_jobs":params.parallel_jobs,"first_keys":model_keys[:5]})


,campaign_id,status,requested_count,created_at_utc,correlation_id
0,39f8c7aa-0e6a-463c-aa77-3748f6d118d5,COMPLETED,1,2026-11-04T20:12:04.656403+00:00,fd9239f4-8408-431d-b8da-0d0cb8befeb8
1,5f1ac7f1-ce17-4ec9-bdaf-53f84853b673,COMPLETED,1,2026-11-04T19:24:41.759118+00:00,785b0ae4-7c88-4054-a2dc-3b3f89e85825
2,7d156255-ee18-4fb6-9c09-a7626dd5e1da,COMPLETED,1,2026-11-04T19:24:28.043044+00:00,168e4337-d4b3-4f39-800f-d1b5126dd34b
3,28d5c5e0-e21d-458c-8e0a-3aca7b4acc31,COMPLETED,1,2026-11-04T18:58:33.650308+00:00,3657341e-9cf1-450c-9360-b5e2892d88b1
4,a4fbe969-d59b-4b41-916e-ae821d26107d,COMPLETED,1,2026-11-04T18:13:27.622600+00:00,6868a0a9-7e9f-42a6-977f-b14ad99b5946
5,4e413355-a21c-4c13-9311-0320857de96c,COMPLETED,1,2026-11-04T18:13:14.034299+00:00,fa61f2fa-5744-4326-9beb-0b69bba37e9e


{'model_count': 30,
 'max_parallel_jobs': 2,
 'first_keys': ['hi.demo.campaign.e3b0c44298.000',
  'hi.demo.campaign.e3b0c44298.001',
  'hi.demo.campaign.e3b0c44298.002',
  'hi.demo.campaign.e3b0c44298.003',
  'hi.demo.campaign.e3b0c44298.004']}

2. Controle opt-in: pause/resume/cancel/retry-failed. Cancelamento é cooperativo; itens em execução concluem, pendentes são cancelados.

In [3]:
if params.campaign_action not in ("create","observe"):
    params.authorize()
    if not params.campaign_id: raise ValueError("campaign_id obrigatório")
    evidence["control"]=call(cfg,"POST",f"/v1/training/campaigns/{params.campaign_id}/{params.campaign_action}")
    display(evidence["control"])


3. Criar campanha somente quando campaign_action=create, ou observar o ID escolhido. Estado/checkpoints permanecem no PostgreSQL após fechar o notebook.

In [4]:
if params.campaign_action!="create":
    params=params.model_copy(update={"execute_write":False})
from src.workbench.operations_sdk import campaign
if params.execute_write:
    params.authorize()
    if not params.snapshot_id:
        raise ValueError("Selecione snapshot_id da tabela SQL")
    payload=dict(snapshot_id=str(params.snapshot_id),cycle_code=cfg.cycle,model_keys=model_keys,
        algorithm=params.algorithm,alpha=params.alpha,max_parallel_jobs=params.parallel_jobs,
        requested_by=params.requester,confirm_training=True)
    display({"preflight":payload,"budget":"máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas"})
    submitted=campaign(cfg,payload,params.request_key)
    campaign_id=submitted["campaign_id"]
else:
    campaign_id=str(params.campaign_id) if params.campaign_id else ""
if campaign_id:
    state=await_job(cfg,f"/v1/training/campaigns/{campaign_id}",timeout=7200)
    items=call(cfg,"GET",f"/v1/training/campaigns/{campaign_id}/items?limit=500")
    display(state);display(pd.DataFrame(items))
    evidence.update(campaign=state,items=items)
else:
    evidence["training"]="NOT_EXECUTED: escolha campanha existente ou habilite escrita"


4. Quantidades comprovadas e progresso. Treinos reais, versões e instâncias de serving são números distintos.

In [5]:
items=evidence.get("items",[])
display({"requested":len(items),"completed":sum(i["status"]=="COMPLETED" for i in items),"registered_versions":sum(bool(i["model_version"]) for i in items),"automatically_promoted":0,"new_HTTP_servers":0})


{'requested': 0,
 'completed': 0,
 'registered_versions': 0,
 'automatically_promoted': 0,
 'new_HTTP_servers': 0}

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [6]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v2/05_treinamento_campanha.json'